# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [14]:
%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [15]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [16]:
import os
from glob import glob

ft_dir = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(ft_dir, "**/*.parquet"), recursive = True)
df = dd.read_parquet(parquet_files).set_index("ticker")

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [17]:
dd_feat = df.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = lambda x: x['Close'].shift(1),
        returns = lambda x: x['Close']/x['Close_lag_1'] - 1, 
        hi_lo_range = lambda x: x['High'] - x['Low']
))

C:\Users\vinus\AppData\Local\Temp\ipykernel_31048\2533097921.py:1: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = df.groupby('ticker', group_keys=False).apply(


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
df_pd=dd_feat.compute()
df_pd["rolling_10_avg"]=df_pd.groupby('ticker')['returns'].transform(lambda x: x.rolling(10).mean())

df_pd.head(15)

c:\Users\vinus\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\groupby.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
c:\Users\vinus\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\groupby.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
c:\Users\vinus\miniconda3\envs\dsi_participant\lib\site-pack

,index,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,returns,hi_lo_range,rolling_10_avg
0,5849,2020-01-02,1.85,1.87,1.85,1.86,1.86,36000.0,DVD.csv,DVD,2020,NaN,NaN,0.02,NaN
1,5850,2020-01-03,1.87,1.89,1.86,1.86,1.86,45800.0,DVD.csv,DVD,2020,1.86,0.000000,0.03,NaN
2,5851,2020-01-06,1.89,1.89,1.86,1.88,1.88,25600.0,DVD.csv,DVD,2020,1.86,0.010753,0.03,NaN
3,5852,2020-01-07,1.84,1.87,1.84,1.84,1.84,87400.0,DVD.csv,DVD,2020,1.88,-0.021277,0.03,NaN
4,5853,2020-01-08,1.84,1.88,1.84,1.86,1.86,34100.0,DVD.csv,DVD,2020,1.84,0.010870,0.04,NaN
5,5854,2020-01-09,1.87,1.88,1.83,1.84,1.84,27500.0,DVD.csv,DVD,2020,1.86,-0.010753,0.05,NaN
6,5855,2020-01-10,1.84,1.85,1.80,1.81,1.81,33300.0,DVD.csv,DVD,2020,1.84,-0.016304,0.05,NaN
7,5856,2020-01-13,1.81,1.85,1.70,1.83,1.83,58600.0,DVD.csv,DVD,2020,1.81,0.011050,0.15,NaN
8,5857,2020-01-14,1.84,1.86,1.83,1.85,1.85,23900.0,DVD.csv,DVD,2020,1.83,0.010929,0.03,NaN
9,5858,2020-01-15,1.85,1.90,1.85,1.86,1.86,52000.0,DVD.csv,DVD,2020,1.85,0.005405,0.05,NaN


c:\Users\vinus\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\groupby.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
c:\Users\vinus\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\groupby.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
c:\Users\vinus\miniconda3\envs\dsi_participant\lib\site-pack

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It is not neccessary to convert it. If there is a lot of data, doing it in dask is better. 

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.